# SQL SubQuery
## How to load the dataset(s) to SQL?
Using Python, you can load the large datasets to MySQL database very easily. For that follow the below steps.

- First create a database in your local machine server.

```sql
CREATE DATABASE <database_name>
```

- Next, use Python to load the database

```python
import pandas as pd
from sqlalchemy import create_engine

df = pd.read_csv("file/path/to/the/database.csv")

engine = create_engine("mysql+pymysql://<db_username>:<db_password>@<hostname>/<database_name>")
df.to_sql("<table_name>", con=engine)
```

## Problems 1-6

For problems 1 to 6, use the Olympic dataset. You can get that from [here](https://drive.google.com/file/d/1EGIRBkbQGByJPvCqDtxtTnXv93oGunFp/view?usp=share_link).

**Column description:**
1. ID -> ID of every records to our dataset. It has integer datatype.
2. Name -> Name of the athletes.
3. Sex -> Gender of the athletes.
4. Height -> Height of the athletes
5. Weight -> Weight of the athletes
6. NOC -> In which country, the athletes belong to. This is actually the country code.
7. Year -> In which year, the athlete has participated
8. Sport -> What is the sport name in which the athlete participated.
9. Event -> Event name of the sport
10. Medal -> Which medal the athlege got. If the athlete did not get any medal then this cell is blank.
11. country -> The name of the country.

### Problem 1

Display the names of athletes who won a gold medal in the 2008 Olympics and whose height is greater than the average height of all athletes in the 2008 Olympics.


### Problem 2

Display the names of athletes who won a medal in the sport of basketball in the 2016 Olympics and whose weight is less than the average weight of all athletes who won a medal in the 2016 Olympics.



### Problem 3

Display the names of all athletes who have won a medal in the sport of swimming in both the 2008 and 2016 Olympics.



### Problem 4

Display the names of all countries that have won more than 50 medals in a single year.



### Problem 5

Display the names of all athletes who have won medals in more than one sport in the same year.



### Problem 6

What is the average weight difference between male and female athletes in the Olympics who have won a medal in the same event?

# solutions 

# 1st
SELECT name
FROM homework.athlete_events
WHERE year = 2008
  AND medal = 'Gold'
  AND height >
      (
        SELECT AVG(height)
        FROM homework.athlete_events
        WHERE year = 2008
          AND height IS NOT NULL
      );


# 2nd
SELECT name
FROM homework.athlete_events
WHERE year = 2016
  AND sport = 'Basketball'
  AND medal IS NOT NULL
  AND weight <
      (
        SELECT AVG(weight)
        FROM homework.athlete_events
        WHERE year = 2016
          AND medal IS NOT NULL
          AND weight IS NOT NULL
      );


# 3rd 
SELECT name
FROM homework.athlete_events
WHERE sport = 'Swimming'
  AND medal IS NOT NULL
  AND year IN (2008, 2016)
GROUP BY name
HAVING COUNT(DISTINCT year) = 2;


# 4th 
SELECT nr.region AS country, ae.year, COUNT(*) AS total_medals
FROM homework.athlete_events ae
JOIN homework.noc_regions nr
  ON ae.NOC = nr.NOC
WHERE ae.medal IS NOT NULL
GROUP BY nr.region, ae.year
HAVING COUNT(*) > 50
ORDER BY total_medals DESC;


# 5th
SELECT name, year
FROM homework.athlete_events
WHERE medal IS NOT NULL
GROUP BY name, year
HAVING COUNT(DISTINCT sport) > 1;


# 6th 
SELECT 
    AVG(m.weight - f.weight) AS avg_weight_difference
FROM homework.athlete_events m
JOIN homework.athlete_events f
  ON m.event = f.event
 AND m.year = f.year
WHERE m.sex = 'M'
  AND f.sex = 'F'
  AND m.medal IS NOT NULL
  AND f.medal IS NOT NULL
  AND m.weight IS NOT NULL
  AND f.weight IS NOT NULL;


## Problem 7 - 10

Use the health insurance dataset. You can get the dataset as well as the description of the dataset [here](https://www.kaggle.com/datasets/thedevastator/insurance-claim-analysis-demographic-and-health).

### Problem 7

How many patients have claimed more than the average claim amount for patients who are smokers and have at least one child, and belong to the southeast region?


### Problem 8

How many patients have claimed more than the average claim amount for patients who are not smokers and have a BMI greater than the average BMI for patients who have at least one child?



### Problem 9

How many patients have claimed more than the average claim amount for patients who have a BMI greater than the average BMI for patients who are diabetic, have at least one child, and are from the southwest region?


### Problem 10:

What is the difference in the average claim amount between patients who are smokers and patients who are non-smokers, and have the same BMI and number of children?

# 7
SELECT COUNT(*)
FROM homework.insurance_data
WHERE region = 'southeast'
  AND claim >
      (
        SELECT AVG(claim)
        FROM homework.insurance_data
        WHERE smokers = 'Yes'
          AND children >= 1
      );


# 8
SELECT COUNT(*)
FROM homework.insurance_data
WHERE smokers = 'No'
  AND claim >
      (
        SELECT AVG(claim)
        FROM homework.insurance_data
        WHERE smokers = 'No'
      )
  AND bmi >
      (
        SELECT AVG(bmi)
        FROM homework.insurance_data
        WHERE children >= 1
      );



# 9
SELECT COUNT(*)
FROM homework.insurance_data
WHERE claim >
      (
        SELECT AVG(claim)
        FROM homework.insurance_data
      )
  AND bmi >
      (
        SELECT AVG(bmi)
        FROM homework.insurance_data
        WHERE diabetic = 'Yes'
          AND children >= 1
          AND region = 'southwest'
      );


# 10
